In [1]:
%load_ext autoreload
%autoreload 2
%pdb on # clickable err traceback



from __future__ import absolute_import, division, print_function
import torch
from trainer_endoda3 import Trainer
from options_endoda3 import MonodepthOptions


Incorrect argument. Use on/1, off/0, or nothing for a toggle.


/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/third_party/EndoDAC/models/backbones/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/third_party/EndoDAC/models/backbones/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/third_party/EndoDAC/models/backbones/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [5]:
# Minimal options for testing
options = MonodepthOptions()
args = [
    '--batch_size', '8',
    '--batch_size', '2',
    '--batch_size', '1',
    '--num_workers', '0',
    '--of_samples',
    '--of_samples_num', '1',
    '--of_samples_num', '10',
    '--frame_ids', '0', '-1', '1',
    '--dataset', 'endovis',
    '--data_path', '/mnt/cluster/workspaces/jinjingxu/SCARED_Images_Resized/',
    '--log_dir', '/tmp/endoda_debug',
    '--log_dir', '/mnt/cluster/workspaces/jinjingxu/tmp/',
    '--compute_depth_metrics',
    '--endoda3_model_config', '/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/networks/configs/endo-da3-all-wowrapper.yaml',
    '--pose_model_type', 'da3_internal',
    '--k_model_type', 'da3_internal',
    '--of_supervised_with_which', 'inputs_color',
    '--use_perframe_gt_K',
    '--learn_intrinsics',
    '--train_data_file', 'test_files.txt',
    '--train_data_file', 'test_files_sequence1_val.txt',
    '--val_data_file', 'test_files.txt test_files_sequence1_val.txt  test_files_sequence2_val.txt ',
    '--val_data_file', 'test_files.txt',
    '--val_data_file', 'test_files_sequence1_val.txt',
    '--da3_depth_regression_target', 'disp',
    '--da3_depth_regression_target', 'disp',
    '--da3_depth_regression_target', 'depth2disp',
    '--depth_model_type', 'depthanything3',
    '--depth_model_type', 'endodac',
    '--k_model_type', 'mlp_with_pn_bottleneck_ipt',
    '--warm_up_step', '5000',
    '--pose_model_type', 'separate_resnet',
    '--pretrained_path', 'depth-anything/da3-base',
    '--pretrained_path', '/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/weights/depthanything',
    '--af_model_type', 'adjust_net',
    # '--depth_model_type', 'endodac',

# --depth_model_type endodac --da3_depth_regression_target disp --k_model_type mlp_with_pn_bottleneck_ipt --warm_up_step 5000 --pretrained_path /mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/weights/depthanything \


    # '--enable_seq_inputs',
    # '--of_model_type', 'raft',
    # '--use_raft_multi_iters',
    # '--raft_trainable_modules', 'convnormrelu layer1 layer2_0 ',
    # '--learn_intrinsics',
 
    # '--endoda3_model_config', '/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/networks/configs/endo-da3-depth-wowrapper.yaml'
]
opts = options.parse_notebook(args)


# Initialize trainer
trainer = Trainer(opts)
print(f"Trainer initialized on {trainer.device}")


load pretrained weight from /mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/weights/depthanything/depth_anything_vitb14.pth


EndoDAC load pretrained DA state dict info:
  Missing keys: 156
    Key prefixes (up to 4 levels): ['depth_head.conv_depth_1.head.0', 'depth_head.conv_depth_1.head.2', 'depth_head.conv_depth_1.head.4', 'depth_head.conv_depth_2.head.0', 'depth_head.conv_depth_2.head.2', 'depth_head.conv_depth_2.head.4', 'depth_head.conv_depth_3.head.0', 'depth_head.conv_depth_3.head.2', 'depth_head.conv_depth_3.head.4', 'depth_head.conv_depth_4.head.0', 'depth_head.conv_depth_4.head.2', 'depth_head.conv_depth_4.head.4', 'encoder.blocks.0.mlp', 'encoder.blocks.1.mlp', 'encoder.blocks.10.mlp', 'encoder.blocks.11.mlp', 'encoder.blocks.11.residual_', 'encoder.blocks.2.mlp', 'encoder.blocks.2.residual_', 'encoder.blocks.3.mlp', 'encoder.blocks.4.mlp', 'encoder.blocks.5.mlp', 'encoder.blocks.5.residual_', 'encoder.blocks.6.mlp', 'encoder.blocks.7.mlp', 'encoder.blocks.8.mlp', 'en

In [ ]:
# # loop over trainer.val_loader to check the collate_fn
# for i, val_inputs in enumerate(trainer.val_loader):
#     print("Batch", i)

In [ ]:
# Get sample batch
trainer.step = 0
trainer.set_train()
train_iter = iter(trainer.train_loader)
inputs = next(train_iter)
val_iter = iter(trainer.val_loader)
val_inputs = next(val_iter)

# Forward pass
outputs, losses = trainer.process_batch(inputs)

print("Forward pass completed!")
print(f"Output keys: {len(outputs)} keys")
print(f"Loss: {losses['loss'].item():.6f}")
print(f"Loss components: {list(losses.keys())}")


In [ ]:
# Compute losses explicitly
losses = trainer.compute_losses(inputs, outputs)

print("Loss computation:")
for key, val in losses.items():
    if isinstance(val, torch.Tensor):
        print(f"  {key}: {val.item():.6f}")
    else:
        print(f"  {key}: {val}")

# compute losses_0 explicitly
losses_0 = trainer.compute_losses_0(inputs, outputs)

print("Loss computation_0:")
for key, val in losses_0.items():
    if isinstance(val, torch.Tensor):
        print(f"  {key}: {val.item():.6f}")
    else:
        print(f"  {key}: {val}")

In [ ]:

val_outputs, _ = trainer.process_batch_val(val_inputs)
val_losses = trainer.compute_losses_val(val_inputs, val_outputs)

In [ ]:
# Get validation batch (has GT depth and poses)
trainer.set_eval()
val_iter = iter(trainer.val_loader)
val_inputs = next(val_iter)

# for flag in [True, False]:
    # trainer.replace_with_gt_rel_rotation = flag
    # print('replace_with_gt_rel_rotation: ', flag)

# Forward pass
with torch.no_grad():
    val_outputs, val_losses = trainer.process_batch(val_inputs)

# Compute depth metrics
from utils.metrics import compute_depth_metrics, compute_pose_metrics

depth_metrics = compute_depth_metrics(val_inputs, val_outputs)
pose_metrics = compute_pose_metrics(val_inputs, val_outputs, opts.frame_ids)

print("Depth Metrics:")
if depth_metrics:
    for key, val in depth_metrics.items():
        print(f"  {key}: {val:.6f}")
else:
    print("  No depth metrics (GT depth not available)")

print("\nPose Metrics:")
print(pose_metrics)
if pose_metrics:
    for key, val in pose_metrics.items():
        print(f"  {key}: {val:.6f}")
else:
    print("  No pose metrics (GT poses not available)")


In [ ]:
# Full training step
trainer.set_train()
train_iter = iter(trainer.train_loader)
inputs = next(train_iter)

# Forward
outputs, losses = trainer.process_batch(inputs)

# Backward
trainer.model_optimizer.zero_grad()
losses["loss"].backward()
trainer.model_optimizer.step()

print(f"Training step completed! Loss: {losses['loss'].item():.6f}")
